# Delta Lake Assignment - Incremental Data Processing

**Objective:** Load a customer dataset into a Delta table, clean it, then use
`MERGE` to apply an incremental update (update existing customers + insert
new ones), and validate the result.

## Before You Start

Upload both CSV files to Databricks:

`Catalog -> Volume (or Workspace Upload) -> Upload`

Upload:
- `customer_master.csv`
- `customer_incremental.csv`

This notebook assumes they end up here:

```
/Volumes/workspace/default/data/customer_master.csv
/Volumes/workspace/default/data/customer_incremental.csv
```

If your files are stored in a different location (e.g. DBFS `/FileStore/tables/`
instead of a Unity Catalog Volume), just change the two paths accordingly.


##Import Libraries

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

##Read Customer Master Dataset

In [0]:
master_df = spark.read.csv("/Volumes/workspace/default/data/customer_master.csv",
header=True,inferSchema=True)
display(master_df.limit(20))
print("Total Rows:",master_df.count())

customer_id,first_name,last_name,email,city,state,signup_date,loyalty_points,status,last_updated
1529,Isha,Pillai,isha.pillai1529@example.com,Lucknow,Uttar Pradesh,2022-08-23,4410,Active,2023-01-25
1459,Neha,Agarwal,neha.agarwal1459@example.com,Indore,Madhya Pradesh,2022-08-22,72,Active,2023-02-25
1099,Anjali,Davis,anjali.davis1099@example.com,Bhopal,Madhya Pradesh,2019-10-07,571,Inactive,2020-05-16
1189,Shreya,Kumar,shreya.kumar1189@example.com,Chennai,Tamil Nadu,2020-06-17,3655,Active,2020-11-20
1293,Meera,Wilson,meera.wilson1293@example.com,Kolkata,West Bengal,2022-07-13,1511,Active,2023-03-01
1453,Isha,Menon,isha.menon1453@example.com,Indore,Madhya Pradesh,2021-06-26,1847,Inactive,2022-04-11
1390,Vivaan,Mehta,vivaan.mehta1390@example.com,Indore,Madhya Pradesh,2020-09-12,2547,Inactive,2021-05-31
1052,Meera,Wilson,meera.wilson1052@example.com,Pune,Maharashtra,2019-10-29,200,Active,2020-03-12
1207,Mike,Johnson,mike.johnson1207@example.com,Ahmedabad,Gujarat,2020-08-08,4396,Active,2020-10-12
1010,Arjun,Reddy,arjun.reddy1010@example.com,Bhopal,Madhya Pradesh,2022-03-07,2577,Active,2022-11-27


Total Rows: 720


##Data Cleaning

Remove duplicate records

In [0]:
master_df = master_df.dropDuplicates()
print("After removing duplicates")
print("Total rows:",master_df.count())

After removing duplicates
Total rows: 700


Fill missing values

In [0]:
master_df = master_df.fillna({
    "city":"Unknown",
    "state":"Unknown",
    "email":"Unknown",
    "status":"Unknown"
})

Check for null values

In [0]:
print("Nulls in each column")
master_df.select([count(when(col(c).isNull(), c)).alias(c) for c in master_df.columns]).display()

Nulls in each column


customer_id,first_name,last_name,email,city,state,signup_date,loyalty_points,status,last_updated
0,0,0,0,0,0,0,0,0,0


##Save as Delta Table

In [0]:
# dropping first in case this cell gets re-run, so it doesn't error out
spark.sql("DROP TABLE IF EXISTS customer_master")
master_df.write.format("delta").mode("overwrite").saveAsTable("customer_master")

Verify

In [0]:
display(spark.table("customer_master").limit(20))

customer_id,first_name,last_name,email,city,state,signup_date,loyalty_points,status,last_updated
1459,Neha,Agarwal,neha.agarwal1459@example.com,Indore,Madhya Pradesh,2022-08-22,72,Active,2023-02-25
1207,Mike,Johnson,mike.johnson1207@example.com,Ahmedabad,Gujarat,2020-08-08,4396,Active,2020-10-12
1181,Saanvi,Kumar,saanvi.kumar1181@example.com,Nagpur,Maharashtra,2021-01-23,3579,Inactive,2021-09-15
1443,Deepak,Taylor,deepak.taylor1443@example.com,Lucknow,Uttar Pradesh,2021-09-16,4164,Active,2021-12-23
1522,Riya,Rao,riya.rao1522@example.com,Chennai,Tamil Nadu,2021-08-23,2810,Inactive,2021-12-05
1325,Rahul,Singh,rahul.singh1325@example.com,Kolkata,West Bengal,2022-03-10,3447,Inactive,2022-12-24
1666,Divya,Davis,divya.davis1666@example.com,Mumbai,Maharashtra,2022-09-11,3658,Inactive,2022-12-19
1116,Anjali,Moore,anjali.moore1116@example.com,Chennai,Tamil Nadu,2019-12-03,2105,Inactive,2020-05-12
1016,Ramesh,Miller,ramesh.miller1016@example.com,Mumbai,Maharashtra,2022-01-09,4344,Active,2022-07-03
1399,James,Brown,james.brown1399@example.com,Kochi,Kerala,2020-07-17,3471,Inactive,2021-01-21


##Read Incremental Dataset

In [0]:
incremental_df = spark.read.csv("/Volumes/workspace/default/data/customer_incremental.csv",header=True,inferSchema=True)
display(incremental_df.limit(20))
print("Incremental Rows:", incremental_df.count())

customer_id,first_name,last_name,email,city,state,signup_date,loyalty_points,status,last_updated
1727,Kavya,Anderson,kavya.anderson1727@example.com,Hyderabad,Telangana,2024-06-14,289,Active,2024-06-14
1037,Riya,Bansal,riya.bansal1037@example.com,Jaipur,Rajasthan,2020-05-29,3471,Inactive,2024-06-24
1354,Diya,Nair,diya.nair1354@example.com,Delhi,Delhi,2021-07-02,1923,Inactive,2024-04-01
1138,Shreya,Miller,shreya.miller1138@example.com,Jaipur,Rajasthan,2022-06-27,2781,Active,2024-06-07
1714,Suresh,Gupta,suresh.gupta1714@example.com,Pune,Maharashtra,2024-03-14,184,Active,2024-03-14
1746,Zara,Taylor,zara.taylor1746@example.com,Pune,Maharashtra,2024-05-14,437,Active,2024-05-14
1029,Vihaan,Davis,vihaan.davis1029@example.com,Indore,Madhya Pradesh,2019-04-28,4048,Active,2024-02-28
1730,Diya,Rao,diya.rao1730@example.com,Nagpur,Maharashtra,2024-04-13,91,Active,2024-04-13
1553,Rohan,Joshi,rohan.joshi1553@example.com,Kolkata,West Bengal,2019-07-03,4932,Inactive,2024-04-27
1478,Vivaan,Anderson,vivaan.anderson1478@example.com,Chennai,Tamil Nadu,2022-07-17,2351,Active,2024-03-07


Incremental Rows: 150


##MERGE (Update Existing + Insert New)

In [0]:
deltaTable = DeltaTable.forName(spark, "customer_master")
(deltaTable.alias("target").merge(incremental_df.alias("source"),
"target.customer_id = source.customer_id").whenMatchedUpdate(
set={
"first_name": "source.first_name",
"last_name": "source.last_name",
"email": "source.email",
"city": "source.city",
"state": "source.state",
"signup_date": "source.signup_date",
"loyalty_points": "source.loyalty_points",
"status": "source.status",
"last_updated": "source.last_updated"
}).whenNotMatchedInsertAll().execute())

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

##Validation

Row Count

In [0]:
final_df = spark.table("customer_master")
print("Final Row Count:", final_df.count())

Final Row Count: 750


Duplicate Check

In [0]:
final_df.groupBy("customer_id").count().filter(col("count") > 1).display()

customer_id,count


If the duplicate check returns no rows, the merge maintained unique `customer_id` values.

##Display Final Dataset

In [0]:
display(final_df.limit(20))
final_df.count()

customer_id,first_name,last_name,email,city,state,signup_date,loyalty_points,status,last_updated
1459,Neha,Agarwal,neha.agarwal1459@example.com,Indore,Madhya Pradesh,2022-08-22,72,Active,2023-02-25
1181,Saanvi,Kumar,saanvi.kumar1181@example.com,Nagpur,Maharashtra,2021-01-23,3579,Inactive,2021-09-15
1443,Deepak,Taylor,deepak.taylor1443@example.com,Lucknow,Uttar Pradesh,2021-09-16,4164,Active,2021-12-23
1522,Riya,Rao,riya.rao1522@example.com,Chennai,Tamil Nadu,2021-08-23,2810,Inactive,2021-12-05
1325,Rahul,Singh,rahul.singh1325@example.com,Kolkata,West Bengal,2022-03-10,3447,Inactive,2022-12-24
1666,Divya,Davis,divya.davis1666@example.com,Mumbai,Maharashtra,2022-09-11,3658,Inactive,2022-12-19
1116,Anjali,Moore,anjali.moore1116@example.com,Chennai,Tamil Nadu,2019-12-03,2105,Inactive,2020-05-12
1016,Ramesh,Miller,ramesh.miller1016@example.com,Mumbai,Maharashtra,2022-01-09,4344,Active,2022-07-03
1540,Aarav,Bansal,aarav.bansal1540@example.com,Indore,Madhya Pradesh,2019-10-07,2817,Inactive,2020-06-13
1384,Emma,Kapoor,emma.kapoor1384@example.com,Pune,Maharashtra,2022-07-08,1211,Inactive,2022-12-28


750

##Summary

In [0]:
print("="*40)
print("Delta Lake Assignment Summary")
print("="*40)
print("Master Dataset Rows      :", master_df.count())
print("Incremental Dataset Rows :", incremental_df.count())
print("Final Dataset Rows       :", final_df.count())
duplicates = final_df.groupBy("customer_id").count().filter("count > 1").count()
print("Duplicate Customer IDs   :", duplicates)
print("="*40)

Delta Lake Assignment Summary
Master Dataset Rows      : 700
Incremental Dataset Rows : 150
Final Dataset Rows       : 750
Duplicate Customer IDs   : 0
